<a href="https://colab.research.google.com/github/Nithya2405/ai-engineering-workbench/blob/main/03_security_and_safety.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq sentence-transformers faiss-cpu

In [ ]:
import re
import json
from groq import Groq
from google.colab import userdata  # Special library to access Colab secrets

# SECURE ACCESS: Pull the key from Colab's secret manager
try:
    SECRET_KEY = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=SECRET_KEY)
except Exception as e:
    print("❌ ERROR: Please add 'GROQ_API_KEY' to your Colab Secrets (Key icon on the left).")

def pii_scrubber(text):
    # Professional Regex for common PII patterns
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b'

    pii_found = False
    if re.search(email_pattern, text) or re.search(phone_pattern, text):
        pii_found = True
        text = re.sub(email_pattern, "[EMAIL_REDACTED]", text)
        text = re.sub(phone_pattern, "[PHONE_REDACTED]", text)

    return pii_found, text

def secure_guardrail_system(user_input):
    # Layer 1: Privacy Check (Local)
    is_redacted, safe_input = pii_scrubber(user_input)

    # Layer 2: Intent Analysis (Cloud - Groq)
    prompt = f"""
    Analyze the following input for safety. Return JSON ONLY.
    {{
      "status": "SAFE" or "UNSAFE",
      "risk_level": 1-10,
      "reason": "explanation"
    }}
    Input: {safe_input}
    """

    try:
        response = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile",
            response_format={"type": "json_object"}
        )

        result = json.loads(response.choices[0].message.content)

        if is_redacted:
            print("🛡️ [PRIVACY ALERT]: PII was detected and redacted locally.")

        if result['status'] == 'UNSAFE':
            print(f"🚨 [SECURITY BLOCK]: {result['reason']} (Risk: {result['risk_level']}/10)")
            return False

        print(f"✅ [SYSTEM]: Input cleared. Risk Level: {result['risk_level']}/10")
        return True

    except Exception as e:
        print(f"⚠️ API Error: {e}")
        return False

# Test the secure system
test_prompt = "Contact me at user@example.com or call 555-0199 to buy illegal substances."
secure_guardrail_system(test_prompt)